In [ ]:
# Installd pciutils and Ollama
!sudo apt-get install -y pciutils zstd
!curl -fsSL https://ollama.com/install.sh | sh
!ollama -v

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libpci3 pci.ids
The following NEW packages will be installed:
  libpci3 pci.ids pciutils zstd
0 upgraded, 4 newly installed, 0 to remove and 45 not upgraded.
Need to get 946 kB of archives.
After this operation, 3,276 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 pci.ids all 0.0~2022.01.22-1ubuntu0.1 [251 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libpci3 amd64 1:3.7.0-6 [28.9 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 pciutils amd64 1:3.7.0-6 [63.6 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 946 kB in 1s (1,107 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/D

In [ ]:
# Install pinggy
!pip install pinggy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 85.3 MB/s eta 0:00:00


In [ ]:
# Start the Ollama server in a subprocess and check if Ollama is listening on port 11434
import subprocess
import time
import os

def start_ollama_server():
    subprocess.Popen(
        ["ollama", "serve"]
    )
    time.sleep(2)
    if check_ollama_port():
        print("Serveur Ollama démarré")
    else:
        print("Retrying...")
        time.sleep(1)
        start_ollama_server()

def check_ollama_port(port='11434'):
    try:
        output = subprocess.run(['sudo', 'lsof', '-i', '-P', '-n'],
                              capture_output=True, text=True).stdout
        if f":{port} (LISTEN)" in output:
            return True
        else:
            print(f"⚠️ Ollama pas encore détécté sur le port : {port}")
            return False
    except Exception as e:
        print(f"❌ Erreur: {e}")

start_ollama_server()

Serveur Ollama démarré


In [ ]:
# Install Open WebUI
!pip install open-webui

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.9/90.9 kB 10.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.9/88.9 kB 9.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.0/81.0 kB 9.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 9.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.8/150.8 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.4/469.4 kB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# Start the Open WebUI server on port 8000
import subprocess
import time

subprocess.Popen(
    ["open-webui", "serve", "--port", "8000"]
)

print('Lancement en cours, veuillez patienter ...')

time.sleep(60)

print('Open webui lancé')

Lancement en cours, veuillez patienter ...
Open webui lancé


In [ ]:
# Start a pinggy tunnel for Open WebUI
import subprocess
import re

proc = subprocess.Popen(
    ["ssh", "-p", "443", "-R0:localhost:8000",
     "-o", "StrictHostKeyChecking=no",
     "-o", "ServerAliveInterval=30",
     "free.pinggy.io"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    stdin=subprocess.DEVNULL
)

url = None
for line in proc.stdout:
    clean = re.sub(r'\x1b[^a-zA-Z]*[a-zA-Z]|\x1b.|\x07', '', line.decode("utf-8", errors="ignore")).strip()
    match = re.search(r'https://[^\s]*\.run\.pinggy[^\s]*', clean)
    if match:
        url = match.group(0)
        break

print(f"\n\n\n✅ Lien vers l'IA personnelle non censuré : {url}\n\n")


✅ Lien vers l'IA personnelle non censuré: http://zvzga-35-204-171-39.a.free.pinggy.link
